In [1]:
import numpy as np

# Testing a little

In [17]:
data_1 = np.load("../data/scada_data_1.npz", allow_pickle=True)

In [18]:
print(data_1)

NpzFile '../data/scada_data_1.npz' with keys: sensor_readings, col_desc, sensor_readings_time, flow_unit


In [19]:
print(data_1["flow_unit"])

8


In [20]:
print(data_1["col_desc"])

[['pressure' '10' 'psi']
 ['pressure' '11' 'psi']
 ['pressure' '12' 'psi']
 ['pressure' '13' 'psi']
 ['pressure' '21' 'psi']
 ['pressure' '22' 'psi']
 ['pressure' '23' 'psi']
 ['pressure' '31' 'psi']
 ['pressure' '32' 'psi']
 ['pressure' '9' 'psi']
 ['pressure' '2' 'psi']
 ['flow' '10' 'cubic meter/hr']
 ['flow' '11' 'cubic meter/hr']
 ['flow' '12' 'cubic meter/hr']
 ['flow' '21' 'cubic meter/hr']
 ['flow' '22' 'cubic meter/hr']
 ['flow' '31' 'cubic meter/hr']
 ['flow' '110' 'cubic meter/hr']
 ['flow' '111' 'cubic meter/hr']
 ['flow' '112' 'cubic meter/hr']
 ['flow' '113' 'cubic meter/hr']
 ['flow' '121' 'cubic meter/hr']
 ['flow' '122' 'cubic meter/hr']
 ['flow' '9' 'cubic meter/hr']
 ['bulk_species_node' 'Chlorine @ 10' 'MG']
 ['bulk_species_node' 'Chlorine @ 11' 'MG']
 ['bulk_species_node' 'Chlorine @ 12' 'MG']
 ['bulk_species_node' 'Chlorine @ 13' 'MG']
 ['bulk_species_node' 'Chlorine @ 21' 'MG']
 ['bulk_species_node' 'Chlorine @ 22' 'MG']
 ['bulk_species_node' 'Chlorine @ 23' 'MG'

In [21]:
print(data_1["sensor_readings"])
#sensors 25-33 chlorine
#sensors 34-44 arsenic

[[126.85165  120.19229  117.086006 ...   0.         0.         0.      ]
 [127.79879  121.24273  118.18431  ...   0.         0.         0.      ]
 [128.83162  122.38834  119.31003  ...   0.         0.         0.      ]
 ...
 [133.22299  127.26107  125.39017  ...   0.         0.         0.      ]
 [120.56403  120.56403  125.185265 ...   0.         0.         0.      ]
 [119.99929  119.99929  124.55526  ...   0.         0.         0.      ]]


In [22]:
#get the indices/column numbers of the chlorine, flow, and arsenic sensors in the file

chlorine_sensors_indices = []
arsenic_sensors_indices = []
flow_sensors_indices = []
for i in range(0,len(data_1["col_desc"])):
    if "Chlorine" in data_1["col_desc"][i][1]:
        chlorine_sensors_indices.append(i)
    elif "AsIII" in data_1["col_desc"][i][1]:
        arsenic_sensors_indices.append(i)
    elif "flow" in data_1["col_desc"][i][0]:
        flow_sensors_indices.append(i)


In [23]:
chlorine_sensors_indices

[24, 25, 26, 27, 28, 29, 30, 31, 32]

In [24]:
print(data_1.files)

['sensor_readings', 'col_desc', 'sensor_readings_time', 'flow_unit']


In [25]:
#Get sensor readings of chlorine and flow sensors
cl_first = chlorine_sensors_indices[0]
cl_last = chlorine_sensors_indices[-1]
flow_first = flow_sensors_indices[0]
flow_last = flow_sensors_indices[-1]
cols = np.r_[cl_first:cl_last+1,
             flow_first:flow_last+1]
data_1_X_cl = data_1["sensor_readings"][:,cl_first:cl_last+1]
data_1_X_flow = data_1["sensor_readings"][:, flow_first:flow_last+1]

#first 9 columns in data_1 are chlorine sensors, the rest are flow sensors

In [26]:
#double check that amount of values is right
print(data_1_X_cl[0])
print(data_1_X_flow[0])

[0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 426.45697    307.60675     12.489316    59.691998    21.583134
   10.106865  -286.63446    100.92958    -10.2694235   -2.863221
   20.831396     1.5775098  426.45697  ]


In [27]:
as_first = arsenic_sensors_indices[0]
as_last = arsenic_sensors_indices[-1]
data_1_y = data_1["sensor_readings"][:, as_first:as_last+1]

In [28]:
print(data_1_y[0])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


# Reading the files

In [31]:
data = []

#Iterate over the different numpy files
for i in range(1,21):
    chlorine_sensors = []
    flow_sensors = []
    arsenic_sensors = []
    X_cl = []
    X_flow = []
    y = []

    file_data = np.load(f"../data/scada_data_{i}.npz", allow_pickle=True)

    #Get the indices/column numbers of the chlorine, flow, and arsenic sensors in the file

    for j in range(0,len(file_data["col_desc"])):
        if "Chlorine" in file_data["col_desc"][j][1]:
            chlorine_sensors.append(j)
        elif "AsIII" in file_data["col_desc"][j][1]:
            arsenic_sensors.append(j)
        elif "flow" in file_data["col_desc"][j][0]:
            flow_sensors.append(j)

    #Get sensor readings of chlorine and flow sensors
    cl_first = chlorine_sensors[0]
    cl_last = chlorine_sensors[-1]
    flow_first = flow_sensors[0]
    flow_last = flow_sensors[-1]
    cols = np.r_[cl_first:cl_last+1,
                flow_first:flow_last+1]
    X_cl.append(file_data["sensor_readings"][:, cl_first:cl_last+1])
    X_flow.append(file_data["sensor_readings"][:, flow_first:flow_last+1])

    #Get sensor readings of arsenic sensors
    as_first = arsenic_sensors[0]
    as_last = arsenic_sensors[-1]
    y.append(file_data["sensor_readings"][:, as_first:as_last+1])

    data.append({"file": file_data, 
                 "Cl_sensors": chlorine_sensors, 
                 "Flow_sensors": flow_sensors, 
                 "As_sensors": arsenic_sensors, 
                 "X_cl": X_cl, 
                 "X_flow": X_flow, 
                 "y": y})

# Final remarks

Since all the values of each file are finally saved in various lists, inside a dictionary inside the data list, you have to cal them like this:

data[i]["xxxxxx"][0][j]

i is the file you want (counting starts at 0, so scada_data_1 is in data[0])  
"xxxxxx" is the part of the data you want  
Since every value is inside a list with one entry, you always have to use [0]  
j is the value/row (depending on the object)  
* for "file", "X_cl", "X_flow", and "y", it is the row
* for "Cl_sensors", "Flow_sensors", and "As_sensors", it is the value (there is only 1 row)